# **Licenciatura em Ciências da Computação**

### Aprendizagem Computacional 25/26

# IMDB Sentiment Analysis: Feature Engineering Challenge

## 1. Setup & Data Loading

First, we'll grab the dataset. We'll use a subset (10,000 rows) to keep the processing fast during class.

In [22]:
import pandas as pd
import numpy as np
import re
import string
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.feature_extraction.text import CountVectorizer

# Load dataset (directly from a common source)
url = "https://raw.githubusercontent.com/Ankit152/IMDB-Sentiment-Analysis/master/IMDB-Dataset.csv"
df = pd.read_csv(url).sample(10000, random_state=42) # Subset for speed

# Encode target
df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})

print(f"Dataset loaded with {df.shape[0]} reviews.")
df.head()

Dataset loaded with 10000 reviews.


,review,sentiment,label
33553,I really liked this Summerslam due to the look...,positive,1
9427,Not many television shows appeal to quite as m...,positive,1
199,The film quickly gets to a major chase scene w...,negative,0
12447,Jane Austen would definitely approve of this o...,positive,1
39489,Expectations were somewhat high for me when I ...,negative,0


In [23]:
df['label'].value_counts()

label
1    5039
0    4961
Name: count, dtype: int64

## 2. Baseline

In [24]:
X = df["review"].astype(str)
y = df["label"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Train size:", len(X_train))
print("Test size :", len(X_test))

Train size: 8000
Test size : 2000


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

vectorizer_baseline = CountVectorizer(max_features=1000)  #Bag Of words -> 1_000 features
X_train_baseline_bow = vectorizer_baseline.fit_transform(X_train)


print(f"Shape of X_train_vectorized: {X_train_baseline_bow.shape}")

model = LogisticRegression(max_iter=1000, random_state=42) # Increased max_iter for convergence
model.fit(X_train_baseline_bow, y_train)

X_test_baseline_bow = vectorizer_baseline.transform(X_test)
y_pred_baseline = model.predict(X_test_baseline_bow)

print("Classification Report:")
print(f"F1 Score: {f1_score(y_test, y_pred_baseline)}")

Shape of X_train_vectorized: (8000, 1000)
Classification Report:
F1 Score: 0.8344895936570862


## 3. The Challenge: Feature Engineering Sandbox
Your goal is to create a function that extracts new numerical features from the raw text. Think about:

Metadata: Length, punctuation, capitalization.

Lexicons: Positive/Negative word counts.

Context: Handling "not" or "never."

In [26]:
df.head()

,review,sentiment,label
33553,I really liked this Summerslam due to the look...,positive,1
9427,Not many television shows appeal to quite as m...,positive,1
199,The film quickly gets to a major chase scene w...,negative,0
12447,Jane Austen would definitely approve of this o...,positive,1
39489,Expectations were somewhat high for me when I ...,negative,0


In [27]:
#we are repeating the split just to be clear what to use on the challenge,
#but this will return the same X_train, X_test as above... because of the fixed random state!

X = df["review"].astype(str)
y = df["label"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Train size:", len(X_train))
print("Test size :", len(X_test))

Train size: 8000
Test size : 2000


In [28]:
y_train.value_counts(), y_test.value_counts()

(label
 1    4031
 0    3969
 Name: count, dtype: int64,
 label
 1    1008
 0     992
 Name: count, dtype: int64)

In [29]:
def simple_clean(text):
    # Convert to lowercase
    text = text.lower()
    # Standardize whitespace (replace multiple whitespaces with a single space)
    text = re.sub(r'\s+', ' ', text).strip()
    # remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # remove digits
    text = re.sub(r'\d+', '', text)
    return text


def extract_custom_features(text):
    """
    Students: Edit this function to create your features!
    Return a list or numpy array of numbers.
    """
    features = []

    # Example Feature 1: Word Count
    words = text.split()
    features.append(len(words))

    # Example Feature 2: Count of Exclamation Marks
    features.append(text.count('!'))
    features.append(text.count('?'))
    # Example Feature 3: 'No/Not' density (Simple Negation)
    negations = len(re.findall(r'\b(not|no|never|neither|nor|bad|worst|disappointment|tedious|horrible)\b', text.lower()))
    features.append(negations)
    affirmations = len(re.findall(r'\b(great|excellent|good|surprising)\b', text.lower()))
    features.append(affirmations)
    # --- ADD YOUR OWN IDEAS BELOW ---

    return features


In [33]:
#re-run this after adding new custom features
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack

vectorizer1 = TfidfVectorizer(max_features=5000,     # só as 5000 palavras mais importantes
    ngram_range=(1,2),     # usa uni- e bigramas ("not good")
    stop_words='english',)
vectorizer2 = CountVectorizer() 
X_train_manual_features = pd.DataFrame(X_train.apply(extract_custom_features).tolist())
X_train_cleaned = X_train.apply(simple_clean)
X_train_text_bow = vectorizer2.fit_transform(X_train_cleaned)
X_train_text_itf = vectorizer1.fit_transform(X_train_cleaned)

X_train_final2 = hstack([X_train_text_bow, X_train_manual_features])
X_train_final1 = hstack([X_train_text_itf, X_train_manual_features])
print("First 5 entries of X_train_manual_features:")
print(X_train_manual_features.head())


First 5 entries of X_train_manual_features:
     0  1  2  3  4
0  130  0  0  0  0
1   92  1  0  2  1
2  175  1  0  1  3
3  124  0  0  2  2
4  115  0  0  2  1


In [37]:
from sklearn.feature_selection import SelectKBest, chi2

selector = SelectKBest(chi2, k=3000)  # mantém 3000 melhores features
X_train_selected = selector.fit_transform(X_train_final1, y_train)
from sklearn.naive_bayes import MultinomialNB
#Treinar modelo
mnb = MultinomialNB()
mnb.fit(X_train_selected, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


Additional Tips:

- Check Stop words
- TF-IDF

In [39]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(
    mnb,
    X_train_selected,
    y_train,
    cv=5,
    scoring="f1"
)

print(scores.mean())

0.8202883178581615


In [41]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=1000)

from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, chi2

pipeline = Pipeline([
    ('vectorizer', TfidfVectorizer(max_features=5000, ngram_range=(1,2))),
    ('selector', SelectKBest(chi2, k=3000)),
    ('clf', MultinomialNB())
])

scores = cross_val_score(pipeline, X_train_cleaned, y_train, cv=5, scoring='f1')
print(scores)

[0.85626536 0.86346863 0.85315534 0.86279357 0.83895131]
